# 🎵 Audio Feature Extraction Pipeline

Notebook ini bertujuan untuk mengekstraksi fitur akustik dari sampel audio percakapan Bahasa Jepang antara **native speaker** dan **learner** (pembelajar).

## Tujuan Akhir
Menghasilkan file `features.csv` yang berisi representasi numerik dari setiap sampel audio, yang nantinya dipakai untuk klasifikasi di notebook `modelling.ipynb`.

## Alur Pipeline
1. **Setup & Eksplorasi** — cek library dan sample rate audio.
2. **Definisi Fungsi Ekstraksi MFCC** — fitur spektral berbasis Mel-Frequency Cepstral Coefficients.
3. **Pembuatan Metadata Utterance** — daftar file audio + label.
4. **Preprocessing Audio** — noise reduction, trim, resample, normalisasi.
5. **Definisi Fungsi Ekstraksi Prosodi** — pitch (F0), jitter, shimmer, formant (F1/F2).
6. **Eksekusi Ekstraksi Massal** — loop ke seluruh dataset dan simpan ke CSV.

## Fitur yang Diekstrak

| Kategori | Fitur | Deskripsi Singkat |
|---|---|---|
| **Spektral (MFCC)** | `mfcc1-13_mean/std` | Bentuk spektrum suara (timbre) |
| **Spektral (Delta)** | `delta1-13_mean/std` | Kecepatan perubahan MFCC antar frame |
| **Spektral (Delta-Delta)** | `delta2_1-13_mean/std` | Percepatan perubahan MFCC |
| **Prosodi (F0)** | `f0_mean, f0_std, f0_range` | Nada suara (pitch) |
| **Prosodi (Intonasi)** | `f0_delta_mean_abs, f0_delta_std` | Dinamika perubahan pitch |
| **Kualitas Suara** | `jitter_local, shimmer_local` | Stabilitas getaran pita suara |
| **Formant** | `f1_mean, f2_mean, f1f2_ratio` | Resonansi rongga vokal |
| **Temporal** | `voiced_duration, syllable_rate` | Durasi bicara & ritme suku kata |

## 1️⃣ Setup & Eksplorasi Awal

Sel ini melakukan:
- Instalasi library yang dibutuhkan: `pandas`, `scipy`, `soundfile`.
- Import library utama: `librosa` (audio processing), `numpy`, `matplotlib`.
- Load satu file audio contoh (`learner01_52.wav`) untuk mengecek **sample rate**, **jumlah sample**, dan **durasi**.

**Kenapa perlu cek sample rate?**  
Sample rate menentukan resolusi waktu sinyal. Untuk analisis prosodi dan MFCC, konsistensi SR penting. Pipeline ini akan menyeragamkan semua audio ke **16 kHz**.

In [1]:
# Install library yang dibutuhkan (cukup sekali di awal)
%pip install -q pandas scipy soundfile librosa praat-parselmouth tqdm

from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import librosa.display
import scipy.signal
import soundfile as sf
import matplotlib.pyplot as plt

# Cek satu sampel audio untuk memahami karakteristiknya
audio_path = Path("../data/data_raw/learner01_52.wav")
y, sr = librosa.load(audio_path, sr=None)   # sr=None agar tidak di-resample

print(f"Sample rate  : {sr} Hz")
print(f"Jumlah sample: {len(y)}")
print(f"Durasi       : {len(y) / sr:.2f} detik")


[notice] A new release of pip is available: 23.3.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/home/kopiadem/Documents/MASTER/1. SEM-1/pengenalan-pola/assignment1/jp-native-vs-learner/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sample rate  : 32000 Hz
Jumlah sample: 268800
Durasi       : 8.40 detik


## 2️⃣ Fungsi Ekstraksi Fitur MFCC

MFCC (Mel-Frequency Cepstral Coefficients) adalah representasi **bentuk spektrum suara** yang meniru persepsi pendengaran manusia.

### Tiga Level Turunan MFCC
1. **MFCC** (13 koefisien) — snapshot spektrum per frame.
2. **Delta MFCC** — turunan pertama, mengukur *kecepatan* perubahan spektrum.
3. **Delta-Delta MFCC** — turunan kedua, mengukur *percepatan* perubahan spektrum.

Hasilnya 39 dimensi per frame. Karena frame jumlahnya sangat banyak, kita **agregasi** dengan `mean` dan `std` untuk mendapat 78 fitur per file.

**Catatan:** Tidak digunakan CMVN (Cepstral Mean and Variance Normalization) karena dapat menghapus informasi identitas pembicara yang justru kita butuhkan.

In [2]:
def extract_mfcc_features(path, sr=16000, n_mfcc=13):
    """
    Ekstrak MFCC + Delta + Delta-Delta, lalu agregasi mean & std.
    Return: dict {nama_fitur: nilai}
    """
    # 1. Load & resample ke 16 kHz
    y, _ = librosa.load(path, sr=sr)

    # 2. Buang bagian senyap di awal/akhir (trim silence)
    y, _ = librosa.effects.trim(y, top_db=30)

    # 3. Hitung MFCC dasar -> shape (13, n_frames)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)

    # 4. Turunan pertama (delta) & kedua (delta-delta)
    mfcc_delta  = librosa.feature.delta(mfcc)
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2)

    # 5. Gabungkan jadi 39 koefisien per frame
    mfcc_all = np.vstack([mfcc, mfcc_delta, mfcc_delta2])

    # 6. Agregasi statistik: mean + std (total 78 fitur)
    features = np.concatenate([mfcc_all.mean(axis=1), mfcc_all.std(axis=1)])

    # 7. Bangun nama kolom secara terstruktur
    cols = []
    for f_type in ['mfcc', 'delta', 'delta2']:
        cols.extend([f'{f_type}{i+1}_mean' for i in range(n_mfcc)])
    for f_type in ['mfcc', 'delta', 'delta2']:
        cols.extend([f'{f_type}{i+1}_std' for i in range(n_mfcc)])

    return dict(zip(cols, features))

## 3️⃣ Membangun Metadata Utterance

Sel ini membuat **daftar file audio** yang akan diproses, dengan menggabungkan:
- `metadata.csv` → info pembicara (speaker_id, class, gender, dll).
- Kode utterance (5 kode: `52, 63, 64, 68, 95`) → tiap speaker punya 5 rekaman.

Hasilnya: **60 file audio** (12 speaker × 5 file), dengan 30 label `native` dan 30 label `learner`.

Output: `metadata_utterance.csv` yang menyimpan path file (raw & processed), serta metadata speaker.

In [3]:
# Path folder audio mentah
AUDIO_DIR = Path('../data/data_raw')
EXT = '.wav'

# Baca metadata speaker (12 baris)
meta_speaker = pd.read_csv('../data/metadata.csv')
CODES = ['52', '63', '64', '68', '95']   # 5 utterance per speaker

# Bangun daftar file audio (60 baris)
rows = []
for _, r in meta_speaker.iterrows():
    for code in CODES:
        fpath           = AUDIO_DIR / f"{r['speaker_id']}_{code}{EXT}"
        fpath_processed = Path('../data/data_preprocessed') / f"{r['speaker_id']}_{code}{EXT}"
        if not fpath.exists():
            print(f"⚠️  tidak ketemu: {fpath}")
            continue
        rows.append({
            'file_path':            str(fpath),
            'file_path_processed':  str(fpath_processed),
            'speaker_id':           r['speaker_id'],
            'class':                r['class'],
            'gender':               r['gender'],
            'location':             r['recording_location'],
            'notes':                r['notes'],
        })

meta = pd.DataFrame(rows)
print(f"Total utterance: {meta.shape[0]}")

# Sanity check: kelas harus seimbang (30 vs 30)
assert meta['class'].value_counts().to_dict() == {'learner': 30, 'native': 30}, \
    "Distribusi kelas tidak seimbang!"

meta.to_csv('metadata_utterance.csv', index=False)

Total utterance: 60


## 4️⃣ Preprocessing Audio

Sel ini menerapkan preprocessing **adaptif** bergantung pada kualitas audio.

### Tahapan Preprocessing
1. **Estimasi SNR** — mengukur rasio sinyal suara terhadap noise.
2. **Conditional Noise Reduction**:
   - Jika **SNR < 15 dB** (bising) → pakai `noisereduce` + `preemphasis`.
   - Jika **SNR ≥ 15 dB** (bersih) → cukup trim dengan threshold longgar.
3. **Trim Silence** — potong bagian sunyi di awal/akhir.
4. **Resample ke 16 kHz** — menyeragamkan sample rate.
5. **Peak Normalization** — skala amplitudo ke 0.9 agar konsisten.

Output disimpan ke `../data/data_preprocessed/`.

In [4]:
OUT_DIR = Path('../data/data_preprocessed'); OUT_DIR.mkdir(exist_ok=True)
SR = 16000

def estimate_snr(y):
    """Estimasi SNR (dB) dari perbandingan energi frame tenang vs frame vokal."""
    rms = librosa.feature.rms(y=y)[0]
    noise_rms  = np.percentile(rms, 10)   # 10% frame paling sunyi
    signal_rms = np.percentile(rms, 90)   # 10% frame paling kencang
    if noise_rms == 0 or np.isnan(noise_rms):
        return 100.0
    return 20 * np.log10(signal_rms / noise_rms)

def preprocess(y, sr):
    """Pipeline preprocessing adaptif berdasarkan SNR."""
    snr = estimate_snr(y)

    if snr < 15:
        # Audio bising -> coba noise reduction
        try:
            import noisereduce as nr
            y = nr.reduce_noise(y=y, sr=sr, stationary=True, prop_decrease=0.8)
            top_db = 40   # trim lebih agresif
        except ImportError:
            y = librosa.effects.preemphasis(y)
            top_db = 40
    else:
        top_db = 30   # audio sudah bersih, trim lebih longgar

    # Trim silence
    y, _ = librosa.effects.trim(y, top_db=top_db)

    # Resample ke SR target
    if sr != SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=SR)

    # Peak normalization ke 0.9
    max_amp = np.max(np.abs(y))
    if max_amp > 0:
        y = y * (0.9 / max_amp)

    return y

# Terapkan ke semua file
for _, r in meta.iterrows():
    y, sr = librosa.load(r['file_path'], sr=None)
    y = preprocess(y, sr)
    sf.write(OUT_DIR / Path(r['file_path']).name, y, SR)

print("✅ Preprocessing selesai untuk seluruh file.")

✅ Preprocessing selesai untuk seluruh file.


## 5️⃣ Fungsi Ekstraksi Fitur Prosodi

Fitur **prosodi** menangkap aspek **nada, ritme, dan kualitas suara** — sering menjadi pembeda utama native vs learner (aksen terdengar dari pitch & intonasi).

### Fitur yang Diekstrak

**A. Temporal**
- `voiced_duration`: durasi total suara setelah trim.
- `syllable_rate`: jumlah onset (perkiraan suku kata) / durasi — proxy **speaking rate**.

**B. Pitch / F0** (menggunakan Praat via `parselmouth`)
- `f0_mean`, `f0_std`, `f0_range`.
- `f0_delta_mean_abs`, `f0_delta_std` — dinamika perubahan nada.

**C. Voice Quality**
- `jitter_local`: variasi periodisitas pitch (kualitas getaran pita suara).
- `shimmer_local`: variasi amplitudo antar siklus.

**D. Formant**
- `f1_mean`, `f2_mean`: resonansi rongga mulut (menunjukkan posisi lidah).
- `f1f2_ratio`: rasio F1/F2 — indikator vokal & aksen.

In [5]:
import parselmouth
from parselmouth.praat import call

def extract_prosody(path, sr=16000):
    """Ekstrak fitur prosodi (F0, jitter, shimmer, formant, temporal)."""
    snd = parselmouth.Sound(path)
    feats = {}

    # --- Temporal: voiced duration & syllable rate ---
    y, _ = librosa.load(path, sr=sr)
    y_trimmed, _ = librosa.effects.trim(y, top_db=30)
    feats['voiced_duration'] = len(y_trimmed) / sr

    onset_env = librosa.onset.onset_strength(y=y_trimmed, sr=sr)
    peaks, _ = scipy.signal.find_peaks(onset_env, height=np.mean(onset_env))
    feats['syllable_rate'] = (
        len(peaks) / feats['voiced_duration']
        if feats['voiced_duration'] > 0 else 0
    )

    # --- Pitch (F0) ---
    pitch = call(snd, "To Pitch", 0.0, 75, 600)
    f0 = pitch.selected_array['frequency']
    f0 = f0[f0 > 0]   # hanya frame voiced

    if len(f0):
        feats['f0_mean']  = f0.mean()
        feats['f0_std']   = f0.std()
        feats['f0_range'] = f0.max() - f0.min()
        if len(f0) > 1:
            f0_delta = np.diff(f0)
            feats['f0_delta_mean_abs'] = np.abs(f0_delta).mean()
            feats['f0_delta_std']      = f0_delta.std()
        else:
            feats['f0_delta_mean_abs'] = np.nan
            feats['f0_delta_std']      = np.nan
    else:
        for k in ['f0_mean','f0_std','f0_range','f0_delta_mean_abs','f0_delta_std']:
            feats[k] = np.nan

    # --- Jitter & Shimmer ---
    point = call(snd, "To PointProcess (periodic, cc)", 75, 600)
    feats['jitter_local']  = call(point, "Get jitter (local)",
                                  0, 0, 0.0001, 0.02, 1.3)
    feats['shimmer_local'] = call([point, snd], "Get shimmer (local)",
                                  0, 0, 0.0001, 0.02, 1.3, 1.2)

    # --- Formant F1 & F2 (sampling tiap 50 ms) ---
    formant = call(snd, "To Formant (burg)", 0.0, 5, 5500, 0.025, 50)
    f1_vals, f2_vals = [], []
    for t in np.arange(0.1, snd.duration - 0.1, 0.05):
        f1 = call(formant, "Get value at time", 1, t, 'Hertz', 'Linear')
        f2 = call(formant, "Get value at time", 2, t, 'Hertz', 'Linear')
        if not np.isnan(f1): f1_vals.append(f1)
        if not np.isnan(f2): f2_vals.append(f2)

    feats['f1_mean'] = np.mean(f1_vals) if f1_vals else np.nan
    feats['f2_mean'] = np.mean(f2_vals) if f2_vals else np.nan
    feats['f1f2_ratio'] = (
        feats['f1_mean'] / feats['f2_mean']
        if f1_vals and f2_vals else np.nan
    )

    return feats

## 6️⃣ Eksekusi Ekstraksi Massal

Sel ini adalah **loop utama**: iterasi ke seluruh 60 file audio yang sudah diproses, menggabungkan fitur MFCC + prosodi, menambahkan metadata speaker, dan menyimpan semuanya ke `features.csv`.

Output: tabel dengan **60 baris × ~92 kolom** (78 MFCC + ~14 prosodi + metadata).

In [6]:
from tqdm import tqdm

def extract_all(path):
    """Gabungkan fitur MFCC + prosodi."""
    feats = extract_mfcc_features(path)
    feats.update(extract_prosody(path))
    return feats

records = []
for _, r in tqdm(meta.iterrows(), total=len(meta), desc="Ekstraksi fitur"):
    feats = extract_all(r['file_path_processed'])
    feats.update(r[['speaker_id', 'class', 'gender', 'location']].to_dict())
    records.append(feats)

features = pd.DataFrame(records)
features.to_csv('features.csv', index=False)
print(f"✅ Selesai. Ukuran features.csv: {features.shape}")

Ekstraksi fitur: 100%|██████████| 60/60 [00:18<00:00,  3.17it/s]

✅ Selesai. Ukuran features.csv: (60, 94)


### 🔧 Sel Opsional: Ekstraksi Terpisah (Debugging)

Jika ingin mengecek/mendebug salah satu kategori fitur saja (misal MFCC saja atau prosodi saja), gunakan sel ini. Output-nya juga menyimpan ke `features.csv` dengan struktur yang sama.

> **Catatan:** Sel ini tidak wajib dijalankan jika sel 6 sudah sukses. Keduanya menulis ke file yang sama.

In [7]:
# Ekstraksi terpisah (opsional, untuk debugging)
df_prosody = pd.DataFrame(
    [extract_prosody(r['file_path_processed']) for _, r in meta.iterrows()]
)
df_prosody['speaker_id'] = meta['speaker_id'].values
df_prosody['class']      = meta['class'].values

df_mfcc = pd.DataFrame(
    [extract_mfcc_features(r['file_path_processed']) for _, r in meta.iterrows()]
)
df_mfcc['speaker_id'] = meta['speaker_id'].values
df_mfcc['class']      = meta['class'].values

# Gabung per baris (bukan per speaker_id), karena tiap speaker punya 5 file
features = pd.concat(
    [
        df_mfcc.reset_index(drop=True),
        df_prosody.drop(columns=['speaker_id', 'class']).reset_index(drop=True)
    ],
    axis=1
)

features.to_csv('features.csv', index=False)
print(f"Ukuran features.csv: {features.shape}")

Ukuran features.csv: (60, 92)
